# PrismLab — P4 Multimodal Research Notebook
### CPU-only LFM3-Omni-2B vs LFM3-Audio-2B → LFM3-VL-2B

**Checkpoint:** 31 October 2026 · **Phase:** P4-A–P4-D (P4-E pending) · **Data:** local JSON experiment records

This notebook **does not run inference or change your benchmark**. It reads the existing results and the frozen P4-D manifest, verifies consistency, and builds auditable tables/plots. Run it from your PrismLab repository; it discovers the project root automatically.

**Working conclusions (based on the original runs, to be recomputed from files):** P4-D was **8/10 strict for both architectures**. The native median was **21.042 s**; modular median **21.447 s** of summed ASR and VL inference. Both failed D04/D05 (nonexistent spatial relations). P4-B favored the modular reasoning-off configuration (6/8 vs 4/8). These are **small, exploratory, single-run experiments**, not general model rankings.

In [ ]:
from pathlib import Path
import json, hashlib, statistics, math, re
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, Markdown

CANDIDATES = [Path.cwd(), *Path.cwd().parents, Path.home() / 'Desktop/flowstate/lfm3-prismlab']
ROOT = next((p for p in CANDIDATES if (p / 'data/p4/robustness/manifest.json').exists()), None)
if ROOT is None:
    raise FileNotFoundError('Run this notebook from PrismLab or place it in notebooks/p4/. Missing data/p4/robustness/manifest.json.')
RESULTS = ROOT / 'results/p4'
MANIFEST_PATH = ROOT / 'data/p4/robustness/manifest.json'
manifest = json.loads(MANIFEST_PATH.read_text(encoding='utf-8'))
manifest_hash = hashlib.sha256(MANIFEST_PATH.read_bytes()).hexdigest()
print('Project root:', ROOT)
print('Dataset:', manifest.get('dataset'))
print('Manifest SHA256:', manifest_hash)
print('Manifest review status:', manifest.get('status'))
print('Recorded cases:', len(manifest.get('cases', [])))
assert len(manifest['cases']) == 10

## 1 · Evidence loading & input-integrity checks

Select the most recent timestamped P4-D files that **match the frozen manifest hash**; fail explicitly rather than mixing experiments. Verify each case's expected answer, image hash and audio hash. Check ASR stage linkage to the vision run.

In [ ]:
def get_report(pattern, *, stage=None):
    matches = []
    for path in RESULTS.glob(pattern):
        data = json.loads(path.read_text(encoding='utf-8'))
        if data.get('manifest_sha256') != manifest_hash:
            continue
        if stage is not None and data.get('stage') != stage:
            continue
        if len(data.get('cases', [])) != 10 or 'summary' not in data:
            continue
        matches.append((path, data))
    if not matches:
        raise FileNotFoundError(f'No complete report matching {pattern!r} with frozen dataset {manifest_hash[:12]}')
    return sorted(matches, key=lambda x: x[0].name)[-1]

native_file, native = get_report('robustness_native_*.json')
modular_file, modular = get_report('robustness_modular_vision_*.json', stage='vision')
asr_file, asr = get_report('robustness_modular_asr_*.json', stage='asr')

# Prefer the exact source ASR trace that created this VL report.
source = modular.get('source_asr_report')
if source:
    source_path = Path(source)
    if not source_path.is_file():
        source_path = RESULTS / source_path.name
    if source_path.is_file():
        loaded = json.loads(source_path.read_text(encoding='utf-8'))
        if loaded.get('manifest_sha256') == manifest_hash:
            asr_file, asr = source_path, loaded

expected_ids = [item['id'] for item in manifest['cases']]
assert expected_ids == [f'D{i:02}' for i in range(1, 11)]
for label, report in [('native', native), ('modular', modular), ('ASR', asr)]:
    lookup = {item['id']: item for item in report['cases']}
    assert set(lookup) == set(expected_ids), f'{label}: missing or duplicate case IDs'
    for case in manifest['cases']:
        row = lookup[case['id']]
        for key in ('expected','image_sha256','audio_sha256'):
            assert row[key] == case[key], f'{label} {case["id"]}: mismatch for {key}'

print('Native:', native_file.name)
print('Modular:', modular_file.name)
print('ASR:', asr_file.name)
print('All 10 case IDs, expected answers, and image/audio hashes match the frozen manifest.')

## 2 · Matched outcomes, exact scoring and latencies

**Strict accuracy** uses the original case-insensitive exact-output rule. `None` and `UNKNOWN` are *not* considered equivalent for strict grading. **Do not alter source JSON or recalculate headline scores from a post-hoc semantic rubric.**

In [ ]:
n = {r['id']: r for r in native['cases']}
m = {r['id']: r for r in modular['cases']}
a = {r['id']: r for r in asr['cases']}
rows = []
for item in manifest['cases']:
    cid = item['id']; nr, mr, ar = n[cid], m[cid], a[cid]
    rows.append({
        'Case': cid,
        'Condition': item['condition'],
        'Expected': item['expected'].upper(),
        'Omni output': nr.get('actual'),
        'Omni strict': bool(nr.get('strict_pass')),
        'Omni s': nr.get('latency_s'),
        'ASR transcript': ar.get('transcript'),
        'ASR exact': bool(ar.get('transcript_exact_match')),
        'ASR s': mr.get('asr_latency_s'),
        'VL output': mr.get('answer'),
        'Modular strict': bool(mr.get('strict_pass')),
        'VL s': mr.get('latency_s'),
        'Modular s': mr.get('total_inference_s'),
        'Omni status': nr.get('status'),
        'VL status': mr.get('status'),
    })
df = pd.DataFrame(rows)
display(df[['Case','Condition','Expected','Omni output','Omni strict','VL output','Modular strict','ASR exact']])
assert df['Omni strict'].sum() == native['summary']['strict_pass']
assert df['Modular strict'].sum() == modular['summary']['strict_pass']
print(f"Omni: {df['Omni strict'].sum()}/10 strict | modular: {df['Modular strict'].sum()}/10 strict")
print(f"ASR exact matches: {df['ASR exact'].sum()}/10")

In [ ]:
def med(series):
    values = [float(v) for v in series if pd.notna(v)]
    return round(statistics.median(values), 3) if values else None

summary = pd.DataFrame([
    {'Pathway': 'Native Omni', 'Strict success': f"{int(df['Omni strict'].sum())}/10",
     'Completion success': f"{(df['Omni status']=='ok').sum()}/10",
     'Median inference (s)': med(df['Omni s']),
     'Latency definition': 'one nonstreaming audio+image call'},
    {'Pathway': 'Modular ASR→VL', 'Strict success': f"{int(df['Modular strict'].sum())}/10",
     'Completion success': f"{(df['VL status']=='ok').sum()}/10",
     'Median inference (s)': med(df['Modular s']),
     'Latency definition': 'per-case ASR + VL request latency'},
])
display(summary)
print('Observed median difference, modular minus native:', round(med(df['Modular s'])-med(df['Omni s']), 3), 's')
print('Note: different residency conditions; no robust inference about speed from one pass.')

## 3 · Visual analysis

Case-level accuracy and latency are **paired by dataset ID**, rather than treating the two pathways as separate random samples. Figures only reflect the measured JSON files and should not be interpreted as population estimates.

In [ ]:
fig, ax = plt.subplots(figsize=(10, 4))
x = list(range(len(df)))
ax.bar([v-0.19 for v in x], df['Omni s'], width=0.38, label='Native Omni')
ax.bar([v+0.19 for v in x], df['Modular s'], width=0.38, label='Modular ASR→VL')
ax.set_xticks(x, df['Case']); ax.set_ylabel('Inference time (s)')
ax.set_title('P4-D · Matched CPU inference latencies'); ax.legend(); ax.grid(axis='y', alpha=0.2)
plt.tight_layout(); plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(10, 3.7))
for j, col in enumerate(['Omni strict','Modular strict']):
    y = [j] * len(df)
    ax.scatter(df['Case'], y, s=220, marker='s', c=df[col].map({True: 1, False: 0}),
               cmap='RdYlGn', vmin=0, vmax=1, edgecolors='black', linewidths=0.4)
ax.set_yticks([0,1], ['Native Omni', 'Modular ASR→VL']); ax.set_ylim(-.7, 1.7)
ax.set_title('P4-D · Case-level strict pass (green) / fail (red)')
ax.grid(axis='x', alpha=0.2)
plt.tight_layout(); plt.show()

## 4 · Failure taxonomy

`D04` and `D05` contain **nonexistent spatial relationships**. They should return `UNKNOWN`, not a nearby object's shape. `D06` asks for a missing **object**, a related but distinct absence test. This distinction is critical to interpreting the results.

In [ ]:
groups = {
    'New layout: answerable': ['D01','D02','D03'],
    'Missing relation': ['D04','D05'],
    'Missing object': ['D06'],
    'Blurred image': ['D07'],
    'Noisy audio': ['D08'],
    'Counterfactual': ['D09','D10'],
}
category = []
for label, ids in groups.items():
    subset = df[df['Case'].isin(ids)]
    category.append({'Category': label, 'n':len(ids),
       'Native strict': f"{subset['Omni strict'].sum()}/{len(ids)}",
       'Modular strict': f"{subset['Modular strict'].sum()}/{len(ids)}"})
display(pd.DataFrame(category))
print('Failed by both:')
display(df[(~df['Omni strict']) & (~df['Modular strict'])][
    ['Case', 'Condition','Expected','Omni output','VL output', 'ASR transcript']])

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
labels = ['Omni total', 'Modular total', 'Omni missing-relation', 'Modular missing-relation']
counts = [int(df['Omni strict'].sum()), int(df['Modular strict'].sum()),
          int(df[df['Case'].isin(['D04','D05'])]['Omni strict'].sum()),
          int(df[df['Case'].isin(['D04','D05'])]['Modular strict'].sum())]
denoms = [10,10,2,2]
ax.bar(labels, [100*k/n for k,n in zip(counts,denoms)])
for i, (k,n) in enumerate(zip(counts,denoms)):
    ax.text(i, max(2,100*k/n+2), f'{k}/{n}', ha='center')
ax.set_ylim(0,108); ax.set_ylabel('Strict correctness (%)'); ax.set_title('Overall vs missing-relation performance')
ax.tick_params(axis='x', rotation=15)
plt.tight_layout(); plt.show()

## 5 · Previous controlled spatial experiment (P4-B/C)

P4-A previously passed **4/4** on the 2×2 spoken-question / image-color factorial test; see `results/p4/factorial_2x2.json` if available. P4-B and P4-C use *different, earlier* cases. Their scores should **never be pooled** with P4-D as though all cases were independent or identically distributed.

In [ ]:
spatial = RESULTS / 'spatial_benchmark.json'
on_file = RESULTS / 'modular_reasoning_on_128.json'
off_file = RESULTS / 'modular_reasoning_off_128.json'
prior = []
if spatial.exists():
    obj = json.loads(spatial.read_text())
    recs = obj if isinstance(obj, list) else obj.get('cases', [])
    if recs:
        prior.append({'Experiment':'P4-B native Omni (corrected input)', 'Strict':f"{sum(bool(r.get('strict_pass')) for r in recs)}/{len(recs)}"})
for path, label in [(on_file,'P4-C modular, reasoning ON'), (off_file,'P4-C modular, reasoning OFF')]:
    if path.exists():
        recs = json.loads(path.read_text())
        if isinstance(recs, list) and recs:
            prior.append({'Experiment':label, 'Strict':f"{sum(bool(r.get('strict_pass')) for r in recs)}/{len(recs)}"})
if prior:
    display(pd.DataFrame(prior))
else:
    print("Optional P4-B/C traces not found; see the P4-D analysis.")

## 6 · Reproducibility, limitations & next experiments

**Recorded conditions:** CPU-only inference, 4-bit GGUF models, `temperature=0`; native Omni receives image+audio in one call. Modular performs Audio-2B ASR then VL-2B vision QA. The modular P4-D setup kept *both specialist servers resident*; the earlier native run used *Omni by itself*. **Measured timings exclude model loading**, and system RAM/swap activity was not continuously recorded for each inference.

**Limitations:** one response per case, 10 synthetic cases, five closely related spoken questions, visually simple geometric shapes, one blurred case, one 15 dB audio-noise case and one counterfactual pair. `D04/D05` share a sparse scene. Therefore, 8/10 should not be presented as population-level robustness, and the approximately 0.4 s median difference should not be treated as statistically meaningful. A 10/10 ASR exact-transcript score is not a guarantee of noisy-speech performance across environments.

**Research conclusion:** On this frozen exploratory benchmark, the architectures tie on exact success, and both fail to abstain on nonexistent spatial relations. That motivates relational-negation probes, not further post-hoc prompt tuning on the same ten cases.

**Proposed next phase:** P4-E — short video frames + synchronized audio descriptions/events, with a fresh manifest, time-aligned labels and a control comparing audio-only, video-only, and combined audio-video inputs. Verify actual runtime support before designing a large evaluation.

In [ ]:
# Export an analysis-ready, clearly derived CSV without changing source records.
export_path = RESULTS / 'p4d_matched_analysis.csv'
df.to_csv(export_path, index=False)
print('Exported derived analysis table:', export_path)
print('Source result files were not modified.')